<a href="https://colab.research.google.com/github/Kamya876/Sentiment-Analysis-using-Word-Embeddings-and-a-Neural-Network./blob/main/12_nn_mini_sentiment_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lecture 12: Activity — Using Word Embeddings with a Neural Net
===============

10/27/2026

You'll need the `gensim` and `torch` (recommended python 3.9 - 3.12) packages for this notebook

https://pytorch.org/get-started/locally/

In [ ]:
# If needed, install dependencies in Google Colab.
# Run this cell only if imports in later cells fail.
%pip -q install gensim torch torchinfo

Task 1: Is a word positive or negative?
-----------------

We'll start by trying to train our neural net from last time to recognize if a word is positive or negative based on its word embedding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# seed random number generation so that you can
# track the same numbers as each other
random.seed(3)

In [ ]:
# Google Colab setup
# Upload sentiment_train.txt and sentiment_dev.txt using the Files panel.
# They should then be available at /content/sentiment_train.txt and /content/sentiment_dev.txt.
# If you uploaded them to Google Drive instead, change the paths in the next cell.

In [ ]:
import os
import gensim.downloader

# Use the uploaded files in the Colab session.
TRAIN_FILE = "/content/sentiment_train.txt"
DEV_FILE = "/content/sentiment_dev.txt"

# If running locally and the files are in the notebook's current directory, use:
if not os.path.exists(TRAIN_FILE) and os.path.exists("sentiment_train.txt"):
    TRAIN_FILE = "sentiment_train.txt"
if not os.path.exists(DEV_FILE) and os.path.exists("sentiment_dev.txt"):
    DEV_FILE = "sentiment_dev.txt"

if not os.path.exists(TRAIN_FILE) or not os.path.exists(DEV_FILE):
    raise FileNotFoundError(
        "Could not find sentiment_train.txt and sentiment_dev.txt. "
        "Upload both files to Colab's Files panel or update TRAIN_FILE and DEV_FILE."
    )

print("Training file:", TRAIN_FILE)
print("Development file:", DEV_FILE)

Training file: /content/sentiment_train.txt
Development file: /content/sentiment_dev.txt


In [ ]:
# Load 50-dimensional pretrained GloVe word vectors.
# The first run downloads the model (about 66 MB), so it needs internet access.
word2vec_vectors = gensim.downloader.load("glove-wiki-gigaword-50")
EMBEDDING_SIZE = word2vec_vectors.vector_size
print("Embedding dimensions:", EMBEDDING_SIZE)

In [ ]:
# accessing a specific word vector
print(word2vec_vectors['computer'])
print("Number of word vectors available:", len(word2vec_vectors))

# TODO:
# play around with the most_similar gensim function (https://radimrehurek.com/gensim/models/word2vec.html)

In [ ]:
def load_data(filename: str) -> list:
    """
    Loads data in the file format
    word,label
    word,label

    where labels are integers
    """
    data = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            pieces = line.split(",")
            data.append((pieces[0], int(pieces[1])))
    return data


train = load_data(TRAIN_FILE)
dev = load_data(DEV_FILE)

print("num in train:", len(train))
print("num in dev:", len(dev))
# what does the data look like?
print(train[:5])
print(dev[:5])

In [ ]:
def create_input_array(examples: list, word_vectors) -> tuple:
    """Convert (word, label) examples into embedding, label, and word arrays."""
    labels = []
    X = []
    words = []
    for word, label in examples:
        # Ignore words that are not in the pretrained vocabulary.
        if word in word_vectors:
            X.append(word_vectors[word])
            labels.append(label)
            words.append(word)

    return np.asarray(X, dtype=np.float32), np.asarray(labels, dtype=np.float32).reshape(-1, 1), words

X, y, words = create_input_array(train, word2vec_vectors)
print("Shape of inputs:", X.shape)
print("Example embedding:", X[0])
print("Shape of labels:", y.shape)
print("Example label:", y[0])
print("Example word:", words[0])

You could use this data with the "from scratch" neural net that we looked at earlier, if you wanted to!

Task 2: Using NN libraries
----------------

Now, we'll start looking at `pytorch` (though you could make this classifier manually with previous lecture activity code if you wanted to!)

Take a look at the pytorch tutorial: https://pytorch.org/tutorials/beginner/basics/intro.html

In [ ]:
# !pip install torch torchvision torchinfo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

try:
    from torchinfo import summary
except ImportError:
    summary = None

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [ ]:
HIDDEN_UNITS = 100
EMBEDDING_SIZE = word2vec_vectors.vector_size
EPOCHS = 10

In [ ]:
# Define a feedforward neural network.
class SentimentNetwork(nn.Module):
    def __init__(self, embedding_size: int, hidden_units: int):
        super().__init__()
        self.input_layer = nn.Linear(embedding_size, hidden_units)
        self.hidden_layer = nn.Linear(hidden_units, 1)
        self.relu = nn.ReLU()
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x):
        hidden = self.relu(self.input_layer(x))
        return self.hidden_layer(hidden)

model = SentimentNetwork(EMBEDDING_SIZE, HIDDEN_UNITS)
if summary is not None:
    summary(model)
else:
    print(model)

In [ ]:
# Train the model using stochastic gradient descent.
# Re-running this cell continues training the current model; re-run the model-creation
# cell first if you want a fresh model for a new HIDDEN_UNITS/EPOCHS experiment.
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    indices = np.random.permutation(len(X))
    for i in indices:
        example = torch.tensor(X[i], dtype=torch.float32)
        label = torch.tensor([y[i, 0]], dtype=torch.float32)

        optimizer.zero_grad()
        output = model(example)
        loss = model.loss_fn(output.reshape(1), label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{EPOCHS} - average loss: {epoch_loss / len(X):.4f}")

if summary is not None:
    summary(model)

Epoch 1/10 - average loss: 0.6360
Epoch 2/10 - average loss: 0.5165
Epoch 3/10 - average loss: 0.4120
Epoch 4/10 - average loss: 0.3578
Epoch 5/10 - average loss: 0.3323
Epoch 6/10 - average loss: 0.3197
Epoch 7/10 - average loss: 0.3121
Epoch 8/10 - average loss: 0.3074
Epoch 9/10 - average loss: 0.3037
Epoch 10/10 - average loss: 0.3008


In [ ]:
# Prepare the development set.
X_dev, y_dev, words_dev = create_input_array(dev, word2vec_vectors)
print("Development examples with available embeddings:", len(X_dev))

Development examples with available embeddings: 369


In [ ]:
# Evaluate the model on the development set.
model.eval()
mistakes = []
num_correct = 0
predictions = []

with torch.no_grad():
    for i, example in enumerate(X_dev):
        output = model(torch.tensor(example, dtype=torch.float32))
        probability = torch.sigmoid(output).item()
        prediction = 1 if probability >= 0.5 else 0
        predictions.append(prediction)

        if prediction == int(y_dev[i, 0]):
            num_correct += 1
        else:
            mistakes.append((words_dev[i], int(y_dev[i, 0]), prediction, probability))

accuracy = num_correct / len(X_dev) if len(X_dev) else 0.0
print(f"Accuracy: {accuracy:.4f} ({num_correct}/{len(X_dev)})")
print("Number of mistakes:", len(mistakes))
print("First five mistakes (word, gold label, prediction, probability):")
print(mistakes[:5])

Accuracy: 0.8835 (326/369)
Number of mistakes: 43
First five mistakes (word, gold label, prediction, probability):
[('dispensable', 0, 1, 0.641465961933136), ('costlier', 0, 1, 0.6251844763755798), ('boom', 1, 0, 0.14391224086284637), ('empathize', 1, 0, 0.4517531991004944), ('selective', 1, 0, 0.2180289477109909)]


1. **Effect of hidden units:** Try values smaller than 50 (for example, 10), equal to 50, and larger than 50 (for example, 100). The number of hidden units changes the model's capacity. Fewer units make a simpler model that may underfit; more units can learn more complex patterns, but do not guarantee better development accuracy. Record the accuracy printed by the evaluation cell for each run.

2. **Effect of epochs:** Try changing `EPOCHS` (for example, 5, 10, and 20). More epochs give the model more opportunities to learn, but accuracy may eventually stop improving or get worse on development data due to overfitting. Record the best development accuracy you observe. The exact accuracy depends on the training run and settings.